# Prediccion del Mundial 2026 - Machine Learning + econometria

Notebook auto-actualizable: lee el Excel `Mundial_2026_fuente_datos.xlsx`,
toma los resultados ya cargados como hechos fijos y recalcula las
probabilidades cada vez que se reejecuta (Entorno de ejecucion -> Ejecutar todo).

Flujo profesional de prediccion:

1. Carga y limpieza de datos (48 selecciones, fixture, cuadro final).
2. Construccion de features por partido: rating Elo, ranking y puntos FIFA,
   titulos, apariciones, mejor resultado historico, valor de plantel, edad,
   **puntaje del DT**, **clasificatoria ponderada por confederacion** y
   **proporcion de jugadores en las 5 grandes ligas**.
3. **Auto-calibracion** de los parametros econometricos (nu, lambda_prior) por
   validacion out-of-fold.
4. Entrenamiento de un **zoo de modelos**: Elo probabilistico, Dixon-Coles/Poisson
   y varios modelos de ML con **auto-tuning** de hiperparametros (logit,
   RandomForest, ExtraTrees, GradientBoosting, HistGradientBoosting y, si estan
   disponibles, XGBoost y LightGBM), con calibracion de probabilidades.
5. Evaluacion out-of-fold (log-loss, accuracy, Brier) + **calibracion (ECE)** y
   seleccion de los **3 mejores modelos**; la prediccion 1/X/2 final es un
   **blend ponderado** de esos tres.
6. Simulacion Monte Carlo del torneo (Dixon-Coles como generador de marcadores)
   con desempate oficial FIFA, localia moderada a los anfitriones en
   eliminatorias y **resultados de eliminatorias ya jugados fijados** (los
   equipos eliminados quedan descartados).

Entrega probabilidades, no recomendaciones de apuestas.

## 1. Preparacion del entorno (clona el repo e instala dependencias en Colab)

In [1]:
import os, sys

EN_COLAB = 'google.colab' in sys.modules
REPO = 'ML_prediccion_mundial2026'
REPO_URL = 'https://github.com/santiagoriverti/ML_prediccion_mundial2026.git'

# En Colab: clonar el repo (trae src/ y el Excel) e instalar requirements.
if EN_COLAB:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL}
    %cd {REPO}
    !pip install -q -r requirements.txt

# Localizar la carpeta src/ (funciona tanto en Colab como en local).
for cand in ['src', '../src', os.path.join(REPO, 'src')]:
    if os.path.exists(cand):
        sys.path.insert(0, os.path.abspath(cand))
        RAIZ = os.path.dirname(os.path.abspath(cand))
        break
print('Raiz del proyecto:', RAIZ)

Raiz del proyecto: C:\Users\sriverti\Desktop\INECO\Repositorios\ML_prediccion_mundial2026


## 2. Carga del Excel (siempre la ultima version commiteada)

In [2]:
import glob, urllib.request
import warnings; warnings.filterwarnings('ignore')

# 1o intenta la raw URL del repo (toma el ultimo commit); 2o archivo local;
# 3o (solo Colab) subida manual.
RAW = ('https://raw.githubusercontent.com/santiagoriverti/'
       'ML_prediccion_mundial2026/main/Mundial_2026_fuente_datos.xlsx')
datos_bytes = None
try:
    datos_bytes = urllib.request.urlopen(RAW, timeout=25).read()
    print('OK - Excel cargado desde la raw URL del repo.')
except Exception as e:
    print('La raw URL fallo (', e, '); busco archivo local .xlsx...')
    locales = glob.glob(os.path.join(RAIZ, '*.xlsx')) + glob.glob('*.xlsx')
    if locales:
        datos_bytes = open(locales[0], 'rb').read()
        print('OK - Excel local:', locales[0])
    elif EN_COLAB:
        from google.colab import files
        subido = files.upload()
        datos_bytes = list(subido.values())[0]
        print('OK - Excel subido manualmente.')
    else:
        raise FileNotFoundError('No se encontro el Excel insumo.')

OK - Excel cargado desde la raw URL del repo.


## 3. Lectura y limpieza de datos

Se cargan las 48 selecciones, el fixture de grupos y el cuadro de eliminatorias.
El loader tolera hojas plantilla vacias, normaliza nombres por `Pais` y descarta
la fila de nota al pie.

In [3]:
from data_loader import cargar_datos
from features import imputar_rating_base, construir_dataset_partidos

datos = cargar_datos(datos_bytes)
print('Equipos:', datos.meta['n_equipos'],
      '| Partidos de grupo:', datos.meta['n_partidos_grupo'],
      '| Ya jugados:', datos.meta['n_jugados'],
      '| Sin puntos FIFA (imputados):', datos.meta['n_sin_puntos'])

equipos = imputar_rating_base(datos.equipos)
equipos[['pais', 'grupo', 'confederacion', 'rating_base', 'rating_imputado']] \
    .sort_values('rating_base', ascending=False).head(10)

Equipos: 48 | Partidos de grupo: 72 | Ya jugados: 56 | Sin puntos FIFA (imputados): 0


,pais,grupo,confederacion,rating_base,rating_imputado
28,España,H,UEFA,1797.506458,0
36,Argentina,J,CONMEBOL,1793.656458,0
32,Francia,I,UEFA,1790.336458,0
44,Inglaterra,L,UEFA,1754.446458,0
8,Brasil,C,CONMEBOL,1680.796458,0
40,Portugal,K,UEFA,1680.726458,0
20,Países Bajos,F,UEFA,1676.596458,0
24,Bélgica,G,UEFA,1651.046458,0
16,Alemania,E,UEFA,1644.476458,0
45,Croacia,L,UEFA,1637.226458,0


## 4. Actualizacion del Elo con los resultados ya cargados

Antes de simular, el rating de cada seleccion se mueve con los partidos ya
jugados (los hechos fijan el pronostico), con factor de margen de victoria.

In [4]:
from simulate import actualizar_elo

equipos = actualizar_elo(equipos, datos.fixture, K=32.0)
dataset = construir_dataset_partidos(equipos, datos.fixture)
print('Dataset por partido:', dataset.shape, '| con resultado (jugados):',
      int(dataset['jugado'].sum()))
equipos[['pais', 'rating_base']].sort_values('rating_base', ascending=False).head(8)

Dataset por partido: (72, 22) | con resultado (jugados): 56


,pais,rating_base
36,Argentina,1804.446153
32,Francia,1801.005107
28,España,1791.774863
44,Inglaterra,1756.851950
8,Brasil,1691.339410
20,Países Bajos,1683.680160
40,Portugal,1681.550869
9,Marruecos,1643.808492


## 5. Variables (features) que usa el modelo

Cada partido se describe por la diferencia A-B de los atributos de las dos
selecciones. Estas son las variables que entran a los modelos supervisados:

In [5]:
from features import COLUMNAS_FEATURES
import pandas as pd

descripcion = {
    'd_rating': 'Diferencia de rating Elo (Puntos FIFA + actualizacion por resultados)',
    'd_ranking': 'Diferencia de ranking FIFA (signo invertido: mejor = menor)',
    'd_puntos': 'Diferencia de Puntos FIFA',
    'd_titulos': 'Diferencia de titulos mundiales',
    'd_apariciones': 'Diferencia de apariciones en Mundiales',
    'd_mejor_result': 'Diferencia de mejor resultado historico (ordinal)',
    'd_valor_plantel': 'Diferencia de valor de plantel (Transfermarkt, decenas de EUR M)',
    'd_edad': 'Diferencia de edad promedio del plantel',
    'd_dt': 'Diferencia de puntaje de trayectoria del DT (seleccion + clubes, 0-100)',
    'd_clasif': 'Diferencia de puntaje de clasificatoria ponderado por dificultad de confederacion',
    'd_top5': 'Diferencia de proporcion de jugadores en las 5 grandes ligas',
    'anfitrion': 'Ventaja de localia (1 si A es sede, -1 si B, 0 si no)',
    'altitud': 'Altitud de la sede (placeholder; hoy 0)',
}
pd.DataFrame({'feature': COLUMNAS_FEATURES,
              'descripcion': [descripcion.get(c, '') for c in COLUMNAS_FEATURES]})

,feature,descripcion
0,d_rating,Diferencia de rating Elo (Puntos FIFA + actual...
1,d_ranking,Diferencia de ranking FIFA (signo invertido: m...
2,d_puntos,Diferencia de Puntos FIFA
3,d_titulos,Diferencia de titulos mundiales
4,d_apariciones,Diferencia de apariciones en Mundiales
5,d_mejor_result,Diferencia de mejor resultado historico (ordinal)
6,d_valor_plantel,"Diferencia de valor de plantel (Transfermarkt,..."
7,d_edad,Diferencia de edad promedio del plantel
8,d_dt,Diferencia de puntaje de trayectoria del DT (s...
9,d_clasif,Diferencia de puntaje de clasificatoria ponder...


## 6. Auto-calibracion de parametros + entrenamiento del zoo de modelos

Primero se **auto-calibran** los parametros econometricos (`nu` del Elo y
`lambda_prior` de Dixon-Coles) buscando, por validacion out-of-fold, los valores
que minimizan el log-loss (sin afinar a mano).

Luego se entrena:
- **Dixon-Coles/Poisson** (con el `lambda_prior` calibrado): genera marcadores
  para la simulacion del torneo.
- Un **zoo de modelos de ML** para 1/X/2 con **auto-tuning** de hiperparametros
  (`RandomizedSearchCV`) y calibracion: regresion logistica, RandomForest,
  ExtraTrees, GradientBoosting, HistGradientBoosting y, si estan instalados,
  **XGBoost** y **LightGBM**.

Con muestra chica (pocos partidos jugados) el nucleo robusto sigue siendo
Elo + Dixon-Coles; el ML entra como complemento y la seleccion del paso 7 decide
de forma data-driven cuanto pesa cada uno.

In [6]:
from models import DixonColes, entrenar_modelos_ml, calibrar_parametros

# 1) Auto-calibracion de parametros econometricos (out-of-fold).
cal = calibrar_parametros(dataset, equipos)
NU, LAMBDA = cal['nu'], cal['lambda_prior']
print(f"Parametros auto-calibrados -> nu={NU} | lambda_prior={LAMBDA} "
      f"| log-loss OOF={cal['log_loss']}")

# 2) Dixon-Coles (generador de marcadores) con el lambda calibrado.
dc = DixonColes(equipos, lambda_prior=LAMBDA).entrenar(dataset)

# 3) Zoo de modelos ML con auto-tuning de hiperparametros (una sola vez).
modelos_ml, reporte = entrenar_modelos_ml(dataset, tune=True)
print('\nModelos ML entrenados:', list(modelos_ml.keys()))
print('Log-loss CV por modelo:', {k: round(v, 3) for k, v in reporte['cv'].items()})
print('Hiperparametros elegidos (auto-tuning):')
for k, v in reporte['hiperparams'].items():
    print('   ', k, '->', v)

Parametros auto-calibrados -> nu=0.2 | lambda_prior=16.0 | log-loss OOF=0.9304



Modelos ML entrenados: ['logit', 'rf', 'extra', 'gbm', 'hist', 'xgb']
Log-loss CV por modelo: {'logit': 0.894, 'rf': 0.878, 'extra': 0.842, 'gbm': 0.874, 'hist': 1.082, 'xgb': 0.877}
Hiperparametros elegidos (auto-tuning):
    logit -> {'logisticregression__C': np.float64(0.02938027938703535)}
    rf -> {'n_estimators': 400, 'min_samples_leaf': 3, 'max_depth': None}
    extra -> {'n_estimators': 200, 'min_samples_leaf': 1, 'max_depth': 3}
    gbm -> {'n_estimators': 100, 'max_depth': 1, 'learning_rate': 0.02}
    hist -> {'min_samples_leaf': 20, 'max_iter': 200, 'max_depth': 2, 'learning_rate': 0.03}
    xgb -> {'subsample': 0.8, 'reg_lambda': 1.0, 'n_estimators': 100, 'min_child_weight': 5, 'max_depth': 2, 'learning_rate': 0.05}


## 7. Evaluacion out-of-fold y seleccion de los 3 mejores modelos

Se compara cada modelo 1/X/2 (Elo, Dixon-Coles y todo el zoo de ML) por
validacion cruzada **out-of-fold** sobre los partidos ya jugados: en cada fold se
reentrenan Dixon-Coles y los modelos ML solo con el train (con los hiperparametros
ya elegidos) y se predice el test, sin fuga de informacion.

La **prediccion final** se hace con los **3 mejores modelos** (menor log-loss),
combinados en un **blend ponderado** (mas peso al de menor log-loss). Asi no se
apuesta todo a un solo modelo: se promedian los tres mas confiables.

Nota: la simulacion del torneo necesita un generador de marcadores, rol que
cumple Dixon-Coles; la seleccion top-3 aplica al pronostico 1/X/2.

In [7]:
from models import evaluar_modelos, seleccionar_top

# Evaluacion OOF (reusa los hiperparametros ya buscados y los parametros
# econometricos calibrados). devolver_oof=True entrega las predicciones OOF
# (oof) y los resultados reales (y_eval) para medir la calibracion mas abajo.
tabla_eval, mejor, oof, y_eval = evaluar_modelos(
    dataset, equipos, devolver_oof=True, nu=NU, lambda_prior=LAMBDA,
    hiperparams=reporte['hiperparams'])

# Los 3 mejores modelos base (excluye 'ensemble') y sus pesos para el blend.
top3, pesos_top = seleccionar_top(tabla_eval, k=3)
print('Los 3 mejores modelos (prediccion final):', top3)
print('Pesos del blend:', {k: round(v, 3) for k, v in pesos_top.items()})
tabla_eval

Los 3 mejores modelos (prediccion final): ['gbm', 'rf', 'extra']
Pesos del blend: {'gbm': 0.336, 'rf': 0.332, 'extra': 0.332}


,modelo,log_loss,accuracy,brier
0,ensemble,0.9336,0.6607,0.5301
1,gbm,0.9434,0.5893,0.5419
2,rf,0.9521,0.6607,0.5158
3,extra,0.9536,0.6250,0.5095
4,logit,0.9704,0.6071,0.5326
5,elo,0.9777,0.6071,0.5434
6,hist,0.9960,0.5714,0.5809
7,xgb,1.0072,0.6071,0.5269
8,dc,1.0192,0.5000,0.6289


## 7b. Calibracion del pronostico (backtesting)

Mas alla de cual modelo acierta mas, conviene saber si sus probabilidades son
**confiables**: cuando dice "55%", ?ocurre cerca del 55% de las veces? Con las
mismas predicciones out-of-fold (sin fuga) se arma un *reliability diagram* en
formato uno-contra-resto: se agrupan las probabilidades en tramos y se compara
la probabilidad media predicha (`conf`) con la frecuencia observada
(`frec_obs`). El **ECE** (Expected Calibration Error, menor = mejor) resume la
brecha promedio.

In [8]:
from models import tabla_calibracion, blend_1x2
import numpy as np

# ECE de cada modelo con prediccion out-of-fold valida (menor = mejor calibrado).
resumen_calib = []
for m, P in oof.items():
    _, ece_m = tabla_calibracion(P, y_eval)
    if ece_m == ece_m:  # descarta NaN (modelos sin oof completo)
        resumen_calib.append({'modelo': m, 'ECE': ece_m})
resumen_calib = pd.DataFrame(resumen_calib).sort_values('ECE').reset_index(drop=True)
print('Calibracion por modelo (ECE, menor = mejor):')
print(resumen_calib.to_string(index=False))

# Reliability del BLEND de los 3 mejores (lo que se usa para predecir).
oof_blend = np.full_like(oof[top3[0]], np.nan)
for i in range(len(y_eval)):
    probs_i = {m: oof[m][i] for m in top3 if m in oof}
    oof_blend[i] = blend_1x2(probs_i, pesos_top)
tabla_calib, ece = tabla_calibracion(oof_blend, y_eval)
print('Reliability del blend top-3', top3, ' | ECE =', ece)
tabla_calib

Calibracion por modelo (ECE, menor = mejor):
  modelo    ECE
    hist 0.0132
     gbm 0.0366
      dc 0.0399
   logit 0.0465
     xgb 0.0466
      rf 0.0479
   extra 0.0634
ensemble 0.0816
     elo 0.1119
Reliability del blend top-3 ['gbm', 'rf', 'extra']  | ECE = 0.0536


,bin_lo,bin_hi,n,conf,frec_obs,gap
0,0.0,0.1,9,0.0943,0.1111,0.0168
1,0.1,0.2,31,0.1393,0.0645,-0.0748
2,0.2,0.3,62,0.2522,0.2419,-0.0102
3,0.3,0.4,15,0.3501,0.4000,0.0499
4,0.4,0.5,10,0.4545,0.3000,-0.1545
5,0.5,0.6,10,0.5700,0.8000,0.2300
6,0.6,0.7,31,0.6356,0.6774,0.0419


## 8. Pronostico por partido pendiente (blend de los 3 mejores)

Para cada partido aun no jugado: P(1/X/2) del **blend de los 3 mejores modelos**,
goles esperados (Dixon-Coles) y marcador mas probable.

In [9]:
from models import pronostico_partidos

tabla_partidos = pronostico_partidos(dataset, equipos, dc, modelos_ml,
                                     modelos_top=top3, pesos_top=pesos_top, nu=NU)
print('Partidos pendientes de jugar:', len(tabla_partidos),
      '| prediccion = blend top-3:', top3)
tabla_partidos.head(20)

Partidos pendientes de jugar: 16 | prediccion = blend top-3: ['gbm', 'rf', 'extra']


,grupo,jornada,equipo_a,equipo_b,P(1),P(X),P(2),goles_esp_A,goles_esp_B,marcador_prob
0,D,3,Estados Unidos,Turquía,0.607,0.237,0.156,2.27,1.40,2-1
1,D,3,Paraguay,Australia,0.590,0.234,0.177,1.22,1.29,1-1
2,F,3,Países Bajos,Túnez,0.657,0.223,0.120,2.02,1.44,1-1
3,F,3,Japón,Suecia,0.514,0.257,0.228,1.70,1.52,1-1
4,G,3,Bélgica,Nueva Zelanda,0.597,0.292,0.111,1.28,1.36,1-1
5,G,3,Egipto,Irán,0.535,0.235,0.230,1.31,1.39,1-1
6,H,3,España,Uruguay,0.644,0.229,0.127,2.33,2.08,2-2
7,H,3,Cabo Verde,Arabia Saudita,0.546,0.240,0.214,1.02,0.91,0-0
8,I,3,Francia,Noruega,0.666,0.221,0.114,2.20,2.15,2-2
9,I,3,Senegal,Irak,0.662,0.217,0.121,1.37,1.25,1-1


## 9. Simulacion Monte Carlo del torneo

Completa los partidos no jugados, resuelve los grupos con el desempate oficial
FIFA, asigna los 8 mejores terceros, arma el cuadro y simula hasta la final
(prorroga/penales por fuerza). Los anfitriones reciben localia moderada en
eliminatorias. Los partidos ya jugados quedan fijos.

In [10]:
from simulate import simular_torneo

N_SIMS = 20000
resultados = simular_torneo(equipos, datos.fixture, datos.bracket, dc,
                            n_sims=N_SIMS, semilla=2026)
print('Corridas:', resultados['n_sims'])

  ... 5000/20000 corridas


  ... 10000/20000 corridas


  ... 15000/20000 corridas


  ... 20000/20000 corridas
Corridas: 20000


## 10. Ranking de probabilidad de ser campeon

In [11]:
tabla_campeon = resultados['campeon'].copy()
tabla_campeon['prob_campeon_%'] = (tabla_campeon['prob_campeon'] * 100).round(2)
tabla_campeon[['pais', 'prob_campeon_%']].head(20)

,pais,prob_campeon_%
0,Estados Unidos,7.22
1,Argentina,7.04
2,México,6.44
3,Francia,6.31
4,España,5.38
5,Brasil,5.25
6,Portugal,5.01
7,Alemania,5.00
8,Inglaterra,4.51
9,Suiza,4.15


## 11. Probabilidad de alcanzar cada ronda

In [12]:
cols_ronda = ['pais', 'prob_32avos', 'prob_16avos', 'prob_Cuartos',
              'prob_Semifinales', 'prob_Final', 'prob_campeon']
av = resultados['avance'][cols_ronda].copy()
for c in cols_ronda[1:]:
    av[c] = (av[c] * 100).round(1)
av.head(16)

,pais,prob_32avos,prob_16avos,prob_Cuartos,prob_Semifinales,prob_Final,prob_campeon
0,Estados Unidos,100.0,65.1,40.6,22.5,12.7,7.2
1,Argentina,100.0,62.2,36.3,21.8,12.2,7.0
2,México,100.0,61.9,37.1,21.1,11.7,6.4
3,Francia,100.0,62.3,37.2,21.3,11.8,6.3
4,España,100.0,58.4,31.9,18.0,10.0,5.4
5,Brasil,100.0,54.8,32.0,17.2,9.5,5.2
6,Portugal,100.0,57.8,30.8,17.4,9.7,5.0
7,Alemania,100.0,60.2,31.8,17.3,9.4,5.0
8,Inglaterra,100.0,55.8,28.9,15.6,8.4,4.5
9,Suiza,100.0,56.4,32.8,16.5,8.4,4.1


## 12. Cuadro de eliminatorias del escenario mas probable

Escenario unico y consistente (cada seleccion aparece una sola vez): completa
los partidos de grupo pendientes con su marcador esperado y resuelve el cuadro
de 32avos con nombres de seleccion. Es una proyeccion que cambia al recargar
resultados; la simulacion Monte Carlo de arriba mantiene toda la incertidumbre.

In [13]:
from simulate import bracket_mas_probable

bracket = bracket_mas_probable(equipos, datos.fixture, datos.bracket, dc)
bracket.columns = ['Partido', 'Equipo 1', 'Equipo 2']
bracket

,Partido,Equipo 1,Equipo 2
0,1.0,Sudáfrica,Canadá
1,2.0,Brasil,Japón
2,3.0,Alemania,Suecia
3,4.0,Países Bajos,Marruecos
4,5.0,Costa de Marfil,Noruega
5,6.0,Francia,Paraguay
6,7.0,México,Ecuador
7,8.0,Inglaterra,Argelia
8,9.0,Nueva Zelanda,Cabo Verde
9,10.0,Estados Unidos,Bosnia y Herzegovina


## 13. Graficos (se guardan en `outputs/`)

In [14]:
from viz import grafico_campeon, heatmap_avance, grafico_calibracion

ruta1 = grafico_campeon(resultados['campeon'], top=15)
ruta2 = heatmap_avance(resultados['avance'], top=20)
ruta3 = grafico_calibracion(tabla_calib, ece, 'blend top-3 ' + '+'.join(top3))
print('Guardado en:', ruta1, ',', ruta2, 'y', ruta3)

Guardado en: outputs\prob_campeon.png , outputs\heatmap_avance.png y outputs\calibracion.png


## 14. Probabilidades por grupo (gana grupo / clasifica)

In [15]:
g = resultados['grupos'].copy()
g['gana_grupo_%'] = (g['prob_gana_grupo'] * 100).round(1)
g['clasifica_%'] = (g['prob_clasifica'] * 100).round(1)
g[['grupo', 'pais', 'gana_grupo_%', 'clasifica_%']]

,grupo,pais,gana_grupo_%,clasifica_%
0,A,México,100.0,100.0
1,A,Sudáfrica,0.0,100.0
2,A,Corea del Sur,0.0,83.3
3,A,Chequia,0.0,0.0
4,B,Canadá,0.0,100.0
5,B,Bosnia y Herzegovina,0.0,100.0
6,B,Suiza,100.0,100.0
7,B,Catar,0.0,0.0
8,C,Brasil,100.0,100.0
9,C,Marruecos,0.0,100.0


## 15. Guardar resultados a `outputs/`

In [16]:
os.makedirs('outputs', exist_ok=True)
tabla_partidos.to_csv('outputs/pronostico_partidos.csv', index=False)
tabla_eval.to_csv('outputs/evaluacion_modelos.csv', index=False)
resumen_calib.to_csv('outputs/calibracion_ece.csv', index=False)
tabla_calib.to_csv('outputs/calibracion_reliability.csv', index=False)
resultados['campeon'].to_csv('outputs/prob_campeon.csv', index=False)
resultados['avance'].to_csv('outputs/prob_avance.csv', index=False)
resultados['grupos'].to_csv('outputs/prob_grupos.csv', index=False)
bracket.to_csv('outputs/bracket_proyectado.csv', index=False)

# Resumen del modelo elegido (parametros calibrados + top-3 + pesos).
import json
with open('outputs/resumen_modelo.json', 'w', encoding='utf-8') as fh:
    json.dump({'nu': NU, 'lambda_prior': LAMBDA,
               'top3': top3, 'pesos_top': pesos_top,
               'hiperparams': reporte['hiperparams']},
              fh, ensure_ascii=False, indent=2, default=float)
print('CSVs y resumen del modelo guardados en outputs/.')

CSVs y resumen del modelo guardados en outputs/.


---
## Como actualizar el pronostico

1. Abri `Mundial_2026_fuente_datos.xlsx` y carga los goles de los partidos
   jugados en la hoja Fixture_Grupos (columnas Goles A / Goles B) y/o en
   Eliminatorias.
2. Commitea y pushea el Excel al repo (la raw URL siempre toma el ultimo commit).
3. Reejecuta el notebook (Ejecutar todo). El rating se actualiza con los nuevos
   resultados, se reentrenan y reevaluan los modelos y los partidos pendientes
   se vuelven a simular.